# Earnings Call Quantitative Backtester & ML Weight Trainer

# 1. Setup, Dependencies & Global Configuration

In [ ]:
%pip install -q datasets transformers sentence-transformers yfinance nltk scikit-learn scipy pandas numpy plotly

import gc
import os
import re
import time
import json
import logging
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import nltk
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from datasets import load_dataset
from nltk.tokenize import sent_tokenize
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, mean_squared_error

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logging.getLogger("yfinance").setLevel(logging.CRITICAL)  # Skryje hlučné 404 chyby u delistovaných tickerů
logger = logging.getLogger("EarningsBacktestML")

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = 0 if torch.cuda.is_available() else -1
DEVICE_STR = "cuda" if DEVICE == 0 else "cpu"
print(f"Setup complete. Running on: {'GPU (CUDA)' if DEVICE == 0 else 'CPU'}")
if DEVICE == 0:
    print(f"GPU Device: {torch.cuda.get_device_name(0)} | Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Setup complete. Running on: GPU (CUDA)
GPU Device: NVIDIA L4 | Total VRAM: 23.7 GB


In [ ]:
USE_DRIVE = False  # Přepněte na True v Colabu pro trvalé ukládání checkpointů na Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/TA2_Earnings_Backtest_v1")
else:
    BASE_DIR = Path("./TA2_Earnings_Backtest_v1")

OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- GLOBÁLNÍ PARAMETRY BACKTESTU & SPLITU ---
TARGET_CALLS = 4000
SEED = 42
RESUME_FROM_CACHE = True
CHECKPOINT_EVERY = 25

TRAIN_RATIO = 0.70
PURGE_GAP_DAYS = 7
TAIL_QUANTILE = 0.20

CACHE_PATH = OUT_DIR / f"backtest_features_{TARGET_CALLS}.csv"
print(f"Outputs & checkpoints directory: {OUT_DIR.resolve()}")
print(f"Config: TARGET_CALLS={TARGET_CALLS} | Split={int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)} | Purge={PURGE_GAP_DAYS}d | Tail={int(TAIL_QUANTILE*100)}%")

Outputs & checkpoints directory: /content/TA2_Earnings_Backtest_v1/outputs
Config: TARGET_CALLS=4000 | Split=70/30 | Purge=7d | Tail=20%


# 2. Data Loading & Hardened Hybrid Parser (Regex + DeBERTa NLI)

In [ ]:
ds = load_dataset("glopardo/sp500-earnings-transcripts")["train"]
print(f"Loaded S&P 500 dataset: {len(ds):,} earnings call transcripts.")
print("Columns:", ds.column_names)

Loaded S&P 500 dataset: 20,681 earnings call transcripts.
Columns: ['ticker', 'company', 'cik', 'sector', 'industry', 'headquarters', 'founded', 'date_added', 'year', 'quarter', 'datacqtr', 'datafqtr', 'earnings_date', 'eps12mtrailing_qavg', 'eps12mtrailing_eoq', 'eps12mfwd_qavg', 'eps12mfwd_eoq', 'eps_lt', 'peforw_qavg', 'peforw_eoq', 'transcript']


In [ ]:
SAFE_HARBOR_PATTERNS = re.compile(
    r"(?:"
    r"forward[-\s]looking statements?|"
    r"private securities litigation reform act|"
    r"safe harbor|"
    r"actual results (?:may|could|might) differ materially|"
    r"risks and uncertainties|"
    r"filings with the (?:sec|securities and exchange commission)|"
    r"form 10-k|form 10-q|"
    r"undertake no obligation to update|"
    r"non-gaap financial measures?"
    r")",
    re.IGNORECASE,
)


def remove_safe_harbor(text: str) -> str:
    if not text or not isinstance(text, str):
        return ""
    sentences = sent_tokenize(text)
    return " ".join(s for s in sentences if not SAFE_HARBOR_PATTERNS.search(s)).strip()


def make_call_id(meta: dict) -> str:
    ticker = re.sub(r"[^A-Za-z0-9_-]", "", str(meta.get("ticker", "UNK")))
    quarter = re.sub(r"[^A-Za-z0-9_-]", "", str(meta.get("datacqtr", "UNK")))
    date_part = re.sub(r"[^0-9]", "", str(meta.get("earnings_date", ""))[:10])
    return f"{ticker}_{quarter}_{date_part}" if date_part else f"{ticker}_{quarter}"

In [ ]:
ROSTER_HEADER_RE = re.compile(
    r"(?:Executives|Analysts|Corporate Participants|Conference Call Participants|Call Participants)\s*:",
    re.IGNORECASE,
)
ANALYST_HEADER_RE = re.compile(
    r"(?:Analysts|Conference Call Participants|Call Participants)\s*:",
    re.IGNORECASE,
)
QA_CUE_RE = re.compile(
    r"(?:"
    r"first question|next question|"
    r"open (?:the (?:call|line|lines|floor|conference) )?(?:for|to|up for) questions|"
    r"begin (?:the )?(?:question[-\s]and[-\s]answer|q&a)|"
    r"proceed (?:with|to) (?:the )?(?:question[-\s]and[-\s]answer|q&a)|"
    r"turn (?:the call )?(?:over )?to (?:the )?(?:question[-\s]and[-\s]answer|q&a)|"
    r"ready (?:for|to take) (?:your )?questions|"
    r"now take (?:your )?(?:first )?questions?|"
    r"question[-\s]and[-\s]answer session (?:will now|begins|is now)|"
    r"our first caller|go to (?:the |our )?(?:first|next) (?:question|caller|line)"
    r")",
    re.IGNORECASE,
)
NON_SPEAKER_PREFIXES = {
    "note", "summary", "operator instructions", "question", "answer", "update",
    "fiscal", "quarter", "slide", "chart", "table", "thank you", "good morning",
    "good afternoon", "good evening", "ladies and gentlemen",
}
ROLE_LABELS = [
    "company executive discussing their own company's financials, operations, or strategy",
    "wall street analyst asking questions about the company",
]


def _clean_candidate_speakers(body_part: str) -> List[str]:
    raw_names = re.findall(
        r"(?:^|[.?!\n]\s*|\]\s+)([A-Z][a-zA-ZÀ-ÿ\'\.-]+(?:\s+[A-Z][a-zA-ZÀ-ÿ\'\.-]+){1,3})\s*(?:-\s*[^:\n]{1,60}?)?\s*:",
        body_part,
    )
    cleaned, seen = [], set()
    for n in raw_names:
        n_clean = re.sub(r"\s+", " ", n).strip(" .,-")
        low = n_clean.lower()
        if not n_clean or low in seen or low in NON_SPEAKER_PREFIXES or len(n_clean) < 4 or len(n_clean) > 45:
            continue
        seen.add(low)
        cleaned.append(n_clean)
    return cleaned


def _refine_inferred_roles_with_nli(df: pd.DataFrame) -> pd.DataFrame:
    """Uses DeBERTa NLI to disambiguate Executive vs. Analyst per speaker when transcript lacks a roster."""
    non_op = df[df["role"] != "Operator"]
    speakers = non_op["speaker"].unique().tolist()
    if not speakers:
        return df

    snippets = [
        " ".join(" ".join(non_op[non_op["speaker"] == sp]["text"].tolist()).split()[:150])
        for sp in speakers
    ]
    preds = nli_pipe(snippets, ROLE_LABELS, multi_label=False, batch_size=16)
    if isinstance(preds, dict):
        preds = [preds]

    role_map = {
        sp: ("Executive" if res["labels"][0] == ROLE_LABELS[0] else "Analyst")
        for sp, res in zip(speakers, preds)
    }
    df = df.copy()
    df.loc[df["role"] != "Operator", "role"] = df.loc[df["role"] != "Operator", "speaker"].map(role_map)

    analyst_idxs = df.index[df["role"] == "Analyst"].tolist()
    if analyst_idxs:
        df.loc[analyst_idxs[0]:, "section"] = "Q&A"
    return df


def parse_transcript(text: str, strip_safe_harbor: bool = True) -> pd.DataFrame:
    if not text or not isinstance(text, str):
        return pd.DataFrame()

    text = text.lstrip("\ufeff").strip()
    if len(text) < 200:
        return pd.DataFrame()

    op_match = re.search(r"\bOperator\s*:", text)
    roster_part, body_part = "", text
    if op_match:
        before_op = text[: op_match.start()].strip()
        if before_op and (ROSTER_HEADER_RE.search(before_op) or (len(before_op) < 1800 and len(sent_tokenize(before_op)) <= 4)):
            roster_part = before_op
            body_part = text[op_match.start() :]

    if roster_part and ANALYST_HEADER_RE.search(roster_part):
        exec_part, analyst_part = ANALYST_HEADER_RE.split(roster_part, maxsplit=1)
    else:
        exec_part, analyst_part = roster_part, ""

    candidate_names = _clean_candidate_speakers(body_part)
    exec_lower, analyst_lower = exec_part.lower(), analyst_part.lower()

    exec_names, analyst_names = [], []
    if roster_part:
        for name in candidate_names:
            if name.lower() in exec_lower:
                exec_names.append(name)
            elif name.lower() in analyst_lower:
                analyst_names.append(name)

    inferred_mode = not (exec_names or analyst_names)
    all_detected = candidate_names if inferred_mode else (exec_names + analyst_names)
    if not all_detected:
        return pd.DataFrame()

    all_speakers = sorted(
        list(dict.fromkeys(all_detected + ["Operator", "Moderator", "Unknown Analyst", "Unknown Executive"])),
        key=len,
        reverse=True,
    )
    speaker_pattern = "|".join(re.escape(name) for name in all_speakers)
    turn_regex = re.compile(rf"(?:^|(?<=[.?!\s\]]))({speaker_pattern})(?:\s*-\s*[^:\n]{{1,60}}?)?\s*:(?:\s*:)?")
    matches = list(turn_regex.finditer(body_part))
    if len(matches) < 3:
        return pd.DataFrame()

    turns, in_qa, n_matches = [], False, len(matches)
    for i, match in enumerate(matches):
        speaker = match.group(1).strip()
        start_idx = match.end()
        end_idx = matches[i + 1].start() if i + 1 < n_matches else len(body_part)
        raw_speech = body_part[start_idx:end_idx].strip()
        if not raw_speech:
            continue

        if speaker in ("Operator", "Moderator"):
            role = "Operator"
            if i > 0 and QA_CUE_RE.search(raw_speech):
                in_qa = True
        elif inferred_mode:
            if not in_qa:
                if i >= 2 and (speaker not in exec_names) and ("?" in raw_speech) and len(raw_speech.split()) < 350:
                    in_qa = True
                    role = "Analyst"
                else:
                    if speaker not in exec_names:
                        exec_names.append(speaker)
                    role = "Executive"
                    if i >= 1 and QA_CUE_RE.search(raw_speech):
                        clean_speech = remove_safe_harbor(raw_speech) if strip_safe_harbor else raw_speech
                        if clean_speech:
                            turns.append({
                                "turn_id": len(turns) + 1, "speaker": speaker, "role": role,
                                "section": "Prepared Remarks", "word_count": len(clean_speech.split()),
                                "text": clean_speech,
                            })
                        in_qa = True
                        continue
            else:
                role = "Executive" if speaker in exec_names else "Analyst"
        else:
            if speaker in exec_names or speaker == "Unknown Executive":
                role = "Executive"
            elif speaker in analyst_names or speaker == "Unknown Analyst":
                role = "Analyst"
            else:
                role = "Operator"

        if role == "Analyst":
            in_qa = True

        section = "Q&A" if in_qa else "Prepared Remarks"
        clean_speech = remove_safe_harbor(raw_speech) if (strip_safe_harbor and section == "Prepared Remarks") else raw_speech
        if not clean_speech:
            continue

        turns.append({
            "turn_id": len(turns) + 1, "speaker": speaker, "role": role,
            "section": section, "word_count": len(clean_speech.split()),
            "text": clean_speech,
        })

    df = pd.DataFrame(turns)
    if inferred_mode and not df.empty and "nli_pipe" in globals():
        df = _refine_inferred_roles_with_nli(df)
    return df


def extract_qa_pairs(
    turns_df: pd.DataFrame,
    min_q_words: int = 5,
    min_followup_words: int = 2,
    min_a_words: int = 5,
) -> pd.DataFrame:
    qa_turns = turns_df[turns_df["section"] == "Q&A"]
    if qa_turns.empty:
        return pd.DataFrame()

    merged = []
    for row in qa_turns.itertuples(index=False):
        txt = str(row.text).strip()
        if not txt:
            continue
        if merged and merged[-1]["role"] == row.role and merged[-1]["speaker"] == row.speaker:
            merged[-1]["text"] += " " + txt
        else:
            merged.append({"role": row.role, "speaker": row.speaker, "text": txt})

    pairs, pair_id, thread_id = [], 1, 0
    current_thread_analyst, seen_q_keys, i = None, set(), 0

    while i < len(merged):
        turn = merged[i]
        if turn["role"] == "Operator":
            current_thread_analyst = None
            i += 1
            continue

        if turn["role"] == "Analyst":
            analyst, q_text = turn["speaker"], turn["text"]
            exec_speakers, exec_texts = [], []
            j = i + 1
            while j < len(merged) and merged[j]["role"] == "Executive":
                exec_speakers.append(merged[j]["speaker"])
                exec_texts.append(merged[j]["text"])
                j += 1

            a_text = " ".join(exec_texts).strip()
            is_followup = (analyst == current_thread_analyst)
            q_threshold = min_followup_words if is_followup else min_q_words
            q_key = (analyst.lower(), q_text[:100].lower())
            has_question = ("?" in q_text) or bool(
                re.search(r"\b(?:can you|could you|would you|what|how|why|wonder|curious|question|prospects)\b", q_text, re.I)
            )

            if (
                has_question
                and len(q_text.split()) >= q_threshold
                and len(a_text.split()) >= min_a_words
                and q_key not in seen_q_keys
            ):
                seen_q_keys.add(q_key)
                if not is_followup:
                    thread_id += 1
                    current_thread_analyst = analyst
                pairs.append({
                    "pair_id": pair_id, "thread_id": thread_id, "is_followup": is_followup,
                    "analyst": analyst, "question": q_text,
                    "executives": " -> ".join(dict.fromkeys(exec_speakers)), "answer": a_text,
                })
                pair_id += 1
            i = j
        else:
            i += 1

    return pd.DataFrame(pairs)

# 3. NLP Models, Loughran-McDonald Lexicon & Feature Extraction

In [ ]:
nli_pipe = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=DEVICE,
)

finbert_pipe = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    device=DEVICE,
    truncation=True,
    max_length=512,
)

embedder = SentenceTransformer("BAAI/bge-m3", device=DEVICE_STR)
print("All 3 encoder models loaded cleanly onto", DEVICE_STR)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

All 3 encoder models loaded cleanly onto cuda


In [ ]:
# --- LOUGHRAN-MCDONALD (LM) FINANCIAL LEXICON ENGINE ---
# Pokud do BASE_DIR nahrajete oficiální soubor 'Loughran-McDonald_MasterDictionary_1993-2025.csv' z sraf.nd.edu,
# načte se automaticky celý slovník (~2 355 neg / 354 pos / 297 unc). Jinak použije vestavěný kurátorovaný LM výběr.

LM_POSITIVE_BUILTIN = {
    "able", "accomplish", "accomplished", "accomplishment", "achieve", "achieved", "achievement", "advancing",
    "advantage", "advantaged", "advantageous", "attractive", "benefit", "benefited", "beneficial", "best",
    "better", "bolster", "bolstered", "boom", "boost", "boosted", "breakthrough", "collaborate", "confident",
    "constructive", "creative", "delighted", "deliver", "delivered", "efficiency", "efficient", "empower",
    "enable", "encouraged", "encouraging", "enhance", "enhanced", "enhancement", "enjoy", "enjoyed", "enthusiastic",
    "exceed", "exceeded", "exceeding", "excel", "excellent", "exceptional", "excited", "exciting", "expand",
    "expanded", "expanding", "expansion", "favorable", "favorably", "gain", "gained", "gains", "good", "great",
    "greater", "greatest", "grow", "growing", "growth", "healthy", "highest", "improve", "improved", "improvement",
    "improving", "innovate", "innovation", "innovative", "leader", "leading", "lucrative", "momentum", "opportunities",
    "opportunity", "optimal", "optimistic", "outperform", "outperformed", "outperforming", "pleased", "positive",
    "positively", "profitable", "profitability", "progress", "progressed", "promising", "record", "resilient",
    "resilience", "rewarding", "robust", "solid", "strength", "strengthen", "strengthened", "strong", "stronger",
    "strongest", "succeed", "succeeded", "success", "successful", "successfully", "superior", "surpass", "surpassed",
    "tailwind", "tailwinds", "tremendous", "unmatched", "unprecedented", "upturn", "valuable", "win", "winning", "wins",
}

LM_NEGATIVE_BUILTIN = {
    "adverse", "adversely", "against", "allegation", "attrition", "bad", "bankruptcy", "bottleneck", "bottlenecks",
    "burden", "cancel", "cancellation", "caution", "challenge", "challenged", "challenges", "challenging", "closure",
    "closures", "collapse", "complaint", "concern", "concerned", "concerns", "constrain", "constrained", "constraint",
    "constraints", "contraction", "costly", "crisis", "critical", "cut", "cutback", "damage", "damaged", "dampen",
    "decline", "declined", "declining", "decrease", "decreased", "decreasing", "default", "defect", "deficiency",
    "deficit", "delay", "delayed", "delays", "deteriorate", "deteriorated", "deteriorating", "deterioration",
    "difficult", "difficulties", "difficulty", "dilute", "dilution", "disappoint", "disappointed", "disappointing",
    "disappointment", "disrupt", "disrupted", "disruption", "disruptions", "divestiture", "downgrade", "downturn",
    "downward", "drag", "drop", "dropped", "erode", "eroded", "erosion", "excessive", "fail", "failed", "failing",
    "failure", "fall", "falling", "fewer", "force", "forced", "fraud", "freeze", "headwind", "headwinds", "hurt",
    "impair", "impaired", "impairment", "inability", "inadequate", "inflation", "inflationary", "insolvency",
    "instability", "insufficient", "interrupt", "interruption", "lag", "lagging", "lawsuit", "layoff", "layoffs",
    "litigation", "loss", "losses", "lost", "lower", "miss", "missed", "missing", "negative", "negatively", "obstacle",
    "outage", "overcapacity", "overrun", "penalty", "penalties", "poor", "pressure", "pressured", "pressures",
    "problem", "problems", "recession", "recessionary", "reduce", "reduced", "reduction", "restructuring", "risk",
    "risks", "risky", "severe", "severely", "shortage", "shortages", "shortfall", "shrink", "shrinking", "shutdown",
    "slow", "slowdown", "slowed", "slower", "slowing", "slump", "soft", "soften", "softened", "softness", "stagnant",
    "strain", "strained", "stress", "struggle", "struggling", "suffer", "suffered", "tariff", "tariffs", "terminate",
    "termination", "threat", "tightening", "trouble", "turbulence", "unable", "underperform", "underperformed",
    "unfavorable", "unplanned", "unprofitable", "unstable", "volatile", "volatility", "vulnerable", "weak", "weaken",
    "weakened", "weaker", "weakness", "worse", "worsen", "worsened", "worst", "writedown", "writeoff",
}

LM_UNCERTAINTY_BUILTIN = {
    "almost", "alteration", "ambiguity", "ambiguous", "anticipate", "apparent", "apparently", "appear", "appears",
    "approximate", "approximately", "assume", "assuming", "assumption", "assumptions", "believe", "cautious",
    "cautiously", "conceivable", "conditional", "contingent", "could", "depend", "dependent", "depending", "depends",
    "doubt", "estimate", "estimated", "estimates", "exposure", "fluctuate", "fluctuation", "fluctuations", "forecast",
    "headwind", "headwinds", "hesitant", "indefinite", "indeterminate", "instability", "intangible", "likelihood",
    "likely", "may", "maybe", "might", "modest", "nearly", "pending", "perhaps", "possibilities", "possibility",
    "possible", "possibly", "precaution", "predict", "predictability", "predicted", "preliminary", "presumably",
    "presume", "probable", "probably", "projected", "random", "reassess", "reconsider", "revise", "risk", "risks",
    "risky", "roughly", "seem", "seems", "softness", "sometime", "somewhat", "speculate", "speculation", "suggest",
    "tentative", "uncertain", "uncertainties", "uncertainty", "unclear", "unconfirmed", "undecided", "undefined",
    "unknown", "unpredictable", "unproven", "unresolved", "unsettled", "unsure", "unusual", "variability", "variable",
    "variance", "variation", "visibility", "volatile", "volatility",
}

NEGATION_WORDS = {"no", "not", "none", "neither", "never", "nobody", "nothing", "nowhere", "without", "cannot", "cant", "dont", "doesnt", "didnt", "wont", "wouldnt", "shouldnt", "isnt", "arent", "wasnt", "werent"}


def load_lm_dictionaries(base_dir: Path = BASE_DIR):
    csv_candidates = list(base_dir.glob("*Loughran*McDonald*.csv"))
    if csv_candidates:
        lm_df = pd.read_csv(csv_candidates[0])
        word_col = "Word" if "Word" in lm_df.columns else lm_df.columns[0]
        pos_set = set(lm_df[lm_df["Positive"] > 0][word_col].astype(str).str.lower())
        neg_set = set(lm_df[lm_df["Negative"] > 0][word_col].astype(str).str.lower())
        unc_set = set(lm_df[lm_df["Uncertainty"] > 0][word_col].astype(str).str.lower())
        print(f"Loaded official LM Master Dictionary from {csv_candidates[0].name} (Pos={len(pos_set)}, Neg={len(neg_set)}, Unc={len(unc_set)})")
        return pos_set, neg_set, unc_set
    print(f"Using built-in curated LM Lexicon (Pos={len(LM_POSITIVE_BUILTIN)}, Neg={len(LM_NEGATIVE_BUILTIN)}, Unc={len(LM_UNCERTAINTY_BUILTIN)})")
    return LM_POSITIVE_BUILTIN, LM_NEGATIVE_BUILTIN, LM_UNCERTAINTY_BUILTIN


LM_POS, LM_NEG, LM_UNC = load_lm_dictionaries()


def score_lm_lexicon(text: str) -> dict:
    """Scores financial text using Loughran-McDonald Positive/Negative/Uncertainty counts with 3-word negation flip."""
    tokens = re.findall(r"[a-z]+", str(text).lower())
    n_words = max(1, len(tokens))
    pos_cnt, neg_cnt, unc_cnt = 0, 0, 0

    for idx, tok in enumerate(tokens):
        if tok in LM_UNC:
            unc_cnt += 1
        if tok in LM_NEG:
            neg_cnt += 1
        elif tok in LM_POS:
            # Standard LM (2011) negation rule: negation within 3 words flips Positive -> Negative
            window = tokens[max(0, idx - 3):idx]
            if any(w in NEGATION_WORDS for w in window):
                neg_cnt += 1
            else:
                pos_cnt += 1

    return {
        "lm_net_score": round(((pos_cnt - neg_cnt) / n_words) * 100, 3),
        "lm_neg_pct": round((neg_cnt / n_words) * 100, 3),
        "lm_uncertainty_rate": round((unc_cnt / n_words) * 100, 3),
    }

Using built-in curated LM Lexicon (Pos=124, Neg=195, Unc=103)


In [ ]:
# --- FINBERT & Q&A EVASION ENGINE ---
REFUSAL_REGEX = re.compile(
    r"don['’]t report|not answering it directly|wait until|won['’]t comment|"
    r"not going to give|too early to say|not prepared to|can['’]t quantify",
    re.IGNORECASE,
)
CANDIDATE_LABELS = [
    "directly answers the question with specific facts or numbers",
    "hedges, expresses uncertainty, or only partially answers the question",
    "refuses to answer, defers to a future date, or deflects the question",
]


def classify_finbert_net(net_sentiment: float, pos_thresh: float = 10.0, neg_thresh: float = -5.0) -> str:
    if net_sentiment >= pos_thresh:
        return "positive"
    if net_sentiment <= neg_thresh:
        return "negative"
    return "neutral"


def score_finbert_text(text: str) -> dict:
    sentences = [s for s in sent_tokenize(text) if len(s.split()) >= 4]
    if not sentences:
        return {"net_sentiment": 0.0, "label": "neutral", "n_sentences": 0}

    preds = finbert_pipe(sentences, batch_size=32)
    pos, neg = [], []
    for sent_pred in preds:
        scores = {item["label"].lower(): item["score"] for item in sent_pred}
        pos.append(scores.get("positive", 0.0))
        neg.append(scores.get("negative", 0.0))

    net_sent = round((float(np.mean(pos)) - float(np.mean(neg))) * 100, 2)
    return {
        "net_sentiment": net_sent,
        "label": classify_finbert_net(net_sent),
        "n_sentences": len(sentences),
    }


def score_nli_chunked(q: str, a: str, chunk_words: int = 180, max_q_words: int = 120) -> float:
    q_words = q.split()
    q_trimmed = " ".join(q_words[-max_q_words:]) if len(q_words) > max_q_words else q
    a_words = a.split()
    chunks = [" ".join(a_words[i : i + chunk_words]) for i in range(0, len(a_words), chunk_words)]
    if not chunks:
        return 0.5

    contexts = [f"Analyst Question: {q_trimmed}\nExecutive Answer: {c}" for c in chunks]
    results = nli_pipe(contexts, CANDIDATE_LABELS, multi_label=False, batch_size=16)
    if isinstance(results, dict):
        results = [results]

    probs = []
    for res in results:
        scores = dict(zip(res["labels"], res["scores"]))
        probs.append((scores[CANDIDATE_LABELS[2]] * 1.0) + (scores[CANDIDATE_LABELS[1]] * 0.5))
    return float(min(probs))


def score_qa_exchange(row: pd.Series) -> pd.Series:
    q, a = row["question"], row["answer"]
    a_sentences = [s for s in sent_tokenize(a) if len(s.split()) >= 4] or [a]
    emb_q = embedder.encode(q, convert_to_tensor=True, normalize_embeddings=True)
    emb_a_sents = embedder.encode(a_sentences, convert_to_tensor=True, normalize_embeddings=True)
    semantic_alignment = float(util.cos_sim(emb_q, emb_a_sents).max())

    q_nums = len(re.findall(r"\b\d+(?:\.\d+)?%?|\$\d+", q))
    a_nums = len(re.findall(r"\b\d+(?:\.\d+)?%?|\$\d+", a))
    quant_evasion = 1.0 if (q_nums >= 2 and a_nums == 0) else 0.0

    nli_evasive_prob = score_nli_chunked(q, a)
    explicit_refusal = bool(REFUSAL_REGEX.search(a))

    evasion_score = min(
        100.0,
        round(
            (nli_evasive_prob * 55)
            + ((1.0 - max(0.0, semantic_alignment)) * 25)
            + (quant_evasion * 10)
            + (25.0 if explicit_refusal else 0.0),
            1,
        ),
    )
    if explicit_refusal or evasion_score >= 62:
        tier = "Fully Evasive"
    elif evasion_score >= 38:
        tier = "Intermediate (Hedging/Pivot)"
    else:
        tier = "Direct"

    a_fb = score_finbert_text(a)
    return pd.Series({
        "evasion_tier": tier,
        "evasion_score": evasion_score,
        "semantic_alignment": round(semantic_alignment, 3),
        "answer_finbert_net": a_fb["net_sentiment"],
        "answer_finbert_label": a_fb["label"],
        "answer_finbert_n_sents": a_fb["n_sentences"],
    })


def extract_call_features(row: dict) -> Optional[dict]:
    """Extracts all NLP features (LM Lexicon, FinBERT Whole/Prep/Q&A, Evasion) for one earnings call."""
    turns_df = parse_transcript(row["transcript"], strip_safe_harbor=True)
    if turns_df.empty:
        return None

    qa_df = extract_qa_pairs(turns_df)
    if qa_df.empty:
        return None

    metrics_df = qa_df.apply(score_qa_exchange, axis=1)
    qa_analysis_df = pd.concat([qa_df, metrics_df], axis=1)

    # Executive text splits
    prep_exec_text = " ".join(turns_df[(turns_df["role"] == "Executive") & (turns_df["section"] == "Prepared Remarks")]["text"])
    qa_exec_text = " ".join(qa_analysis_df["answer"].astype(str))
    all_exec_text = f"{prep_exec_text} {qa_exec_text}".strip()

    # 1. Loughran-McDonald Lexicon features (whole executive call + Q&A uncertainty)
    lm_whole = score_lm_lexicon(all_exec_text)
    lm_qa = score_lm_lexicon(qa_exec_text)

    # 2. FinBERT features (single-pass, sentence-weighted)
    prep_fb = score_finbert_text(prep_exec_text)
    qa_weights = qa_analysis_df["answer_finbert_n_sents"].fillna(0).values
    qa_n = int(qa_weights.sum())
    qa_net = float(np.average(qa_analysis_df["answer_finbert_net"].values, weights=qa_weights)) if qa_n > 0 else 0.0
    prep_n = prep_fb["n_sentences"]
    total_n = prep_n + qa_n
    overall_net = ((prep_fb["net_sentiment"] * prep_n) + (qa_net * qa_n)) / total_n if total_n > 0 else 0.0

    # 3. Evasion & Q&A negative density
    total_q = len(qa_analysis_df)
    full_ev = (qa_analysis_df["evasion_tier"] == "Fully Evasive").sum() / total_q
    inter_ev = (qa_analysis_df["evasion_tier"] == "Intermediate (Hedging/Pivot)").sum() / total_q
    weighted_evasion_pct = round(float((full_ev + 0.5 * inter_ev) * 100), 1)
    qa_neg_pct = round(float((qa_analysis_df["answer_finbert_label"] == "negative").mean() * 100), 1)

    return {
        "call_id": make_call_id(row),
        "ticker": str(row.get("ticker", "UNKNOWN")),
        "company": str(row.get("company", "UNKNOWN")),
        "quarter": str(row.get("datacqtr", "UNKNOWN")),
        "earnings_date": str(row.get("earnings_date", "UNKNOWN")),
        "n_qa_pairs": int(total_q),
        # Tier 1: Loughran-McDonald Lexicon
        "lm_net_score": lm_whole["lm_net_score"],
        "lm_neg_pct": lm_whole["lm_neg_pct"],
        "lm_uncertainty_rate": lm_qa["lm_uncertainty_rate"],
        # Tier 2: FinBERT
        "finbert_score": round(overall_net, 2),
        "finbert_prep_score": round(prep_fb["net_sentiment"], 2),
        "finbert_qa_score": round(qa_net, 2),
        "finbert_tone_delta": round(qa_net - prep_fb["net_sentiment"], 2),
        "qa_neg_pct": qa_neg_pct,
        # Behavioral / Evasion
        "evasion_rate_pct": weighted_evasion_pct,
        "avg_semantic_alignment": round(float(qa_analysis_df["semantic_alignment"].mean()), 3),
    }

# 4. Market Returns Engine (`yfinance`) & Checkpointed Feature Collector

In [ ]:
def _extract_price_series(data: pd.DataFrame, field: str, ticker: str) -> Optional[pd.Series]:
    if isinstance(data.columns, pd.MultiIndex):
        if field in data.columns.get_level_values(0) and ticker in data[field].columns:
            return data[field][ticker]
        return None
    return data[field] if field in data.columns else None


def check_actual_market_returns(ticker: str, earnings_date_str: str) -> Optional[dict]:
    """Computes pre-5d excess return (<= earnings_date) and post-earnings returns vs SPY from T+1 Open."""
    start_dt = pd.to_datetime(earnings_date_str, errors="coerce")
    if pd.isna(start_dt):
        return None
    start_dt = start_dt.tz_localize(None) if start_dt.tzinfo is not None else start_dt
    pre_start_dt = start_dt - pd.Timedelta(days=14)
    end_dt = start_dt + pd.Timedelta(days=40)

    data = yf.download(
        [ticker, "SPY"],
        start=pre_start_dt.strftime("%Y-%m-%d"),
        end=end_dt.strftime("%Y-%m-%d"),
        auto_adjust=True,
        progress=False,
    )
    if data is None or data.empty:
        return None

    data.index = data.index.tz_localize(None) if data.index.tz is not None else data.index
    future_positions = np.where(data.index > start_dt)[0]
    if len(future_positions) < 6:
        return None

    entry_pos = int(future_positions[0])
    open_stock_s = _extract_price_series(data, "Open", ticker)
    open_spy_s = _extract_price_series(data, "Open", "SPY")
    close_stock_s = _extract_price_series(data, "Close", ticker)
    close_spy_s = _extract_price_series(data, "Close", "SPY")
    if any(s is None for s in [open_stock_s, open_spy_s, close_stock_s, close_spy_s]):
        return None

    entry_stock, entry_spy = open_stock_s.iloc[entry_pos], open_spy_s.iloc[entry_pos]
    if pd.isna(entry_stock) or pd.isna(entry_spy) or entry_stock <= 0 or entry_spy <= 0:
        return None

    # Pre-earnings 5-trading-day excess return (strictly <= earnings_date)
    pre_positions = np.where(data.index <= start_dt)[0]
    pre_5d_excess = 0.0
    if len(pre_positions) >= 6:
        p_end, p_start = int(pre_positions[-1]), int(pre_positions[-6])
        s_pre = ((close_stock_s.iloc[p_end] / close_stock_s.iloc[p_start]) - 1) * 100
        spy_pre = ((close_spy_s.iloc[p_end] / close_spy_s.iloc[p_start]) - 1) * 100
        if not (pd.isna(s_pre) or pd.isna(spy_pre)):
            pre_5d_excess = round(float(s_pre - spy_pre), 2)

    results = {
        "entry_date": data.index[entry_pos].strftime("%Y-%m-%d"),
        "pre_5d_excess_ret": pre_5d_excess,
    }
    for offset, key in [(0, "T1"), (4, "T5"), (14, "T15")]:
        tpos = entry_pos + offset
        if tpos < len(data):
            cs, cspy = close_stock_s.iloc[tpos], close_spy_s.iloc[tpos]
            if not (pd.isna(cs) or pd.isna(cspy)):
                results[f"actual_{key}_ret"] = round(float(((cs / entry_stock) - 1) * 100), 2)
                results[f"spy_{key}_ret"] = round(float(((cspy / entry_spy) - 1) * 100), 2)

    if results.get("actual_T1_ret") is None or results.get("actual_T5_ret") is None:
        return None
    return results

In [ ]:
# --- CHECKPOINTED BATCH FEATURE COLLECTOR ---
if RESUME_FROM_CACHE and CACHE_PATH.exists():
    features_df = pd.read_csv(CACHE_PATH)
    results_list = features_df.to_dict("records")
    seen_ids = set(features_df["call_id"].astype(str))
    print(f"Loaded checkpoint '{CACHE_PATH.name}' with {len(results_list)}/{TARGET_CALLS} calls.")
else:
    results_list, seen_ids = [], set()

errors_log = []
t0 = time.time()

if len(results_list) < TARGET_CALLS:
    for idx, r in enumerate(ds.shuffle(seed=SEED)):
        if len(results_list) >= TARGET_CALLS:
            break
        cid = make_call_id(r)
        if cid in seen_ids:
            continue
        try:
            # Nejdříve rychlý market check (ušetří GPU čas u delistovaných tickerů!)
            bt = check_actual_market_returns(str(r.get("ticker", "")), str(r.get("earnings_date", "")))
            if not bt:
                errors_log.append({"idx": idx, "ticker": r.get("ticker"), "reason": "missing_market_data"})
                continue

            feats = extract_call_features(r)
            if not feats:
                errors_log.append({"idx": idx, "ticker": r.get("ticker"), "reason": "parse_or_qa_empty"})
                continue

            feats.update(bt)
            results_list.append(feats)
            seen_ids.add(cid)

            if len(results_list) % CHECKPOINT_EVERY == 0 or len(results_list) == TARGET_CALLS:
                pd.DataFrame(results_list).to_csv(CACHE_PATH, index=False)
                logger.info(f"Checkpoint saved: {len(results_list)}/{TARGET_CALLS} calls ({time.time() - t0:.0f}s elapsed)")
        except Exception as e:
            errors_log.append({"idx": idx, "ticker": r.get("ticker"), "reason": f"{type(e).__name__}: {e}"})

features_df = pd.DataFrame(results_list)
features_df.to_csv(CACHE_PATH, index=False)

# 5. Purged Chronological Split & ML Weight Training

In [ ]:
# --- 5.1 PURGED CHRONOLOGICAL SPLIT ---
chrono_df = features_df.copy()
chrono_df["earnings_dt"] = pd.to_datetime(chrono_df["earnings_date"], errors="coerce")
chrono_df = chrono_df.dropna(subset=["earnings_dt"]).sort_values("earnings_dt").reset_index(drop=True)

chrono_df["excess_T1"] = chrono_df["actual_T1_ret"] - chrono_df["spy_T1_ret"]
chrono_df["excess_T5"] = chrono_df["actual_T5_ret"] - chrono_df["spy_T5_ret"]
chrono_df["year"] = chrono_df["earnings_dt"].dt.year

split_idx = int(len(chrono_df) * TRAIN_RATIO)
test_start_dt = chrono_df["earnings_dt"].iloc[split_idx]
purge_cutoff_dt = test_start_dt - pd.Timedelta(days=PURGE_GAP_DAYS)

train_df = chrono_df[chrono_df["earnings_dt"] <= purge_cutoff_dt].copy().reset_index(drop=True)
embargo_df = chrono_df[(chrono_df["earnings_dt"] > purge_cutoff_dt) & (chrono_df["earnings_dt"] < test_start_dt)].copy()
test_df = chrono_df[chrono_df["earnings_dt"] >= test_start_dt].copy().reset_index(drop=True)

print("=== PURGED CHRONOLOGICAL SPLIT SUMMARY ===")
print(f"  Train Set   (n={len(train_df):3d}): {train_df['earnings_dt'].iloc[0].date()} -> {train_df['earnings_dt'].iloc[-1].date()} | Base Rate (excess_T5 > 0): {(train_df['excess_T5'] > 0).mean()*100:.1f}%")
print(f"  Purge Gap   (n={len(embargo_df):3d}): {PURGE_GAP_DAYS}-day embargo before {test_start_dt.date()} (dropped to prevent T+5 label overlap)")
print(f"  OOS Test    (n={len(test_df):3d}): {test_df['earnings_dt'].iloc[0].date()} -> {test_df['earnings_dt'].iloc[-1].date()} | Base Rate (excess_T5 > 0): {(test_df['excess_T5'] > 0).mean()*100:.1f}%")
print("\nCalls per Year (Regime Coverage):", chrono_df["year"].value_counts().sort_index().to_dict())

=== PURGED CHRONOLOGICAL SPLIT SUMMARY ===
  Train Set   (n=2796): 2013-05-21 -> 2017-12-07 | Base Rate (excess_T5 > 0): 49.8%
  Purge Gap   (n=  4): 7-day embargo before 2017-12-20 (dropped to prevent T+5 label overlap)
  OOS Test    (n=1200): 2017-12-20 -> 2025-01-21 | Base Rate (excess_T5 > 0): 47.8%

Calls per Year (Regime Coverage): {2013: 34, 2014: 501, 2015: 758, 2016: 716, 2017: 794, 2018: 776, 2019: 71, 2020: 89, 2021: 76, 2022: 61, 2023: 58, 2024: 63, 2025: 3}


In [ ]:
# --- 5.2 TIER 1 (LM LEXICON) & TIER 2 (CALIBRATED FINBERT) BASELINES ---
def calibrate_zscore_signal(train_series: pd.Series, test_series: pd.Series, z_thresh: float = 0.5) -> List[str]:
    """Fits mean & std strictly on Train and applies frozen thresholds to Test."""
    mu = float(train_series.mean())
    sd = max(1e-6, float(train_series.std()))
    hi, lo = mu + z_thresh * sd, mu - z_thresh * sd
    return [
        "BUY (LONG)" if v >= hi else ("SHORT" if v <= lo else "NEUTRAL / NO TRADE")
        for v in test_series
    ]

test_df["sig_lm_lexicon"] = calibrate_zscore_signal(train_df["lm_net_score"], test_df["lm_net_score"], z_thresh=0.5)
test_df["sig_finbert_whole"] = calibrate_zscore_signal(train_df["finbert_score"], test_df["finbert_score"], z_thresh=0.5)
test_df["sig_finbert_qa"] = calibrate_zscore_signal(train_df["finbert_qa_score"], test_df["finbert_qa_score"], z_thresh=0.5)

print("Baselines calibrated strictly on Train (μ ± 0.5σ) and frozen for OOS Test.")

Baselines calibrated strictly on Train (μ ± 0.5σ) and frozen for OOS Test.


In [ ]:
# --- 5.3 TIER 3: ML WEIGHT TRAINERS (LOGISTIC REGRESSION & RIDGE + TAIL FILTERING) ---
FULL_FEATURES = [
    "finbert_score",
    "finbert_qa_score",
    "finbert_tone_delta",
    "lm_net_score",
    "lm_uncertainty_rate",
    "qa_neg_pct",
    "evasion_rate_pct",
    "pre_5d_excess_ret",
]

COMPACT_FEATURES = [
    "finbert_score",
    "lm_uncertainty_rate",
    "evasion_rate_pct",
    "pre_5d_excess_ret",
]


def train_tail_logreg(tr_df: pd.DataFrame, te_df: pd.DataFrame, feature_cols: List[str], tail_q: float = TAIL_QUANTILE):
    X_tr = tr_df[feature_cols].fillna(0.0)
    y_tr = (tr_df["excess_T5"] > 0).astype(int)
    X_te = te_df[feature_cols].fillna(0.0)

    n_splits = min(5, max(2, len(X_tr) // 25))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    best_c, best_acc = 0.1, -1.0

    for cand_c in [0.005, 0.01, 0.05, 0.1, 0.5, 1.0]:
        accs = []
        for tr_i, va_i in tscv.split(X_tr):
            if y_tr.iloc[tr_i].nunique() < 2:
                continue
            m = make_pipeline(StandardScaler(), LogisticRegression(C=cand_c, max_iter=1000))
            m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i])
            accs.append(accuracy_score(y_tr.iloc[va_i], m.predict(X_tr.iloc[va_i])))
        if accs and float(np.mean(accs)) > best_acc:
            best_acc, best_c = float(np.mean(accs)), cand_c

    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=best_c, max_iter=1000))
    pipe.fit(X_tr, y_tr)

    tr_probs = pipe.predict_proba(X_tr)[:, 1]
    te_probs = pipe.predict_proba(X_te)[:, 1]
    hi_cut = float(np.quantile(tr_probs, 1.0 - tail_q))
    lo_cut = float(np.quantile(tr_probs, tail_q))

    signals = [
        "BUY (LONG)" if p >= hi_cut else ("SHORT" if p <= lo_cut else "NEUTRAL / NO TRADE")
        for p in te_probs
    ]
    weights = dict(zip(feature_cols, np.round(pipe.named_steps["logisticregression"].coef_[0], 4)))
    return signals, te_probs, weights, best_c, best_acc, (lo_cut, hi_cut)


def train_tail_ridge(tr_df: pd.DataFrame, te_df: pd.DataFrame, feature_cols: List[str], tail_q: float = TAIL_QUANTILE):
    X_tr = tr_df[feature_cols].fillna(0.0)
    y_tr = tr_df["excess_T5"].astype(float)
    X_te = te_df[feature_cols].fillna(0.0)

    n_splits = min(5, max(2, len(X_tr) // 25))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    best_alpha, best_mse = 10.0, float("inf")

    for cand_a in [0.5, 1.0, 5.0, 10.0, 50.0, 100.0, 250.0]:
        mses = []
        for tr_i, va_i in tscv.split(X_tr):
            m = make_pipeline(StandardScaler(), Ridge(alpha=cand_a))
            m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i])
            mses.append(mean_squared_error(y_tr.iloc[va_i], m.predict(X_tr.iloc[va_i])))
        if mses and float(np.mean(mses)) < best_mse:
            best_mse, best_alpha = float(np.mean(mses)), cand_a

    pipe = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha))
    pipe.fit(X_tr, y_tr)

    tr_preds = pipe.predict(X_tr)
    te_preds = pipe.predict(X_te)
    hi_cut = float(np.quantile(tr_preds, 1.0 - tail_q))
    lo_cut = float(np.quantile(tr_preds, tail_q))

    signals = [
        "BUY (LONG)" if p >= hi_cut else ("SHORT" if p <= lo_cut else "NEUTRAL / NO TRADE")
        for p in te_preds
    ]
    weights = dict(zip(feature_cols, np.round(pipe.named_steps["ridge"].coef_, 4)))
    return signals, te_preds, weights, best_alpha, (lo_cut, hi_cut)


# Train all 3 ML models strictly on Train
test_df["sig_logreg_full"], test_df["prob_logreg_full"], w_logreg_full, c_full, cv_full, cuts_full = train_tail_logreg(train_df, test_df, FULL_FEATURES)
test_df["sig_logreg_compact"], test_df["prob_logreg_compact"], w_logreg_compact, c_comp, cv_comp, cuts_comp = train_tail_logreg(train_df, test_df, COMPACT_FEATURES)
test_df["sig_ridge_full"], test_df["pred_ridge_full"], w_ridge_full, a_ridge, cuts_ridge = train_tail_ridge(train_df, test_df, FULL_FEATURES)

print(f"Full LogReg (8 features)    | Best C={c_full} | Train CV Acc={cv_full*100:.1f}% | Tail Cutoffs: <= {cuts_full[0]:.3f} / >= {cuts_full[1]:.3f}")
print("  Learned Weights:", w_logreg_full)
print(f"\nCompact LogReg (4 features) | Best C={c_comp} | Train CV Acc={cv_comp*100:.1f}% | Tail Cutoffs: <= {cuts_comp[0]:.3f} / >= {cuts_comp[1]:.3f}")
print("  Learned Weights:", w_logreg_compact)
print(f"\nFull Ridge (8 features)     | Best alpha={a_ridge} | Tail Cutoffs: <= {cuts_ridge[0]:+.2f}% / >= {cuts_ridge[1]:+.2f}%")
print("  Learned Weights:", w_ridge_full)

Full LogReg (8 features)    | Best C=1.0 | Train CV Acc=50.9% | Tail Cutoffs: <= 0.477 / >= 0.519
  Learned Weights: {'finbert_score': np.float64(0.074), 'finbert_qa_score': np.float64(-0.0886), 'finbert_tone_delta': np.float64(0.0266), 'lm_net_score': np.float64(-0.0377), 'lm_uncertainty_rate': np.float64(-0.0327), 'qa_neg_pct': np.float64(0.0437), 'evasion_rate_pct': np.float64(-0.0264), 'pre_5d_excess_ret': np.float64(0.0739)}

Compact LogReg (4 features) | Best C=0.05 | Train CV Acc=50.5% | Tail Cutoffs: <= 0.483 / >= 0.513
  Learned Weights: {'finbert_score': np.float64(-0.0466), 'lm_uncertainty_rate': np.float64(-0.0202), 'evasion_rate_pct': np.float64(-0.0136), 'pre_5d_excess_ret': np.float64(0.0712)}

Full Ridge (8 features)     | Best alpha=250.0 | Tail Cutoffs: <= -0.10% / >= +0.20%
  Learned Weights: {'finbert_score': np.float64(0.0456), 'finbert_qa_score': np.float64(-0.0885), 'finbert_tone_delta': np.float64(0.0429), 'lm_net_score': np.float64(-0.0127), 'lm_uncertainty_rat

# 6. Out-of-Sample Scorecard, Downside Risk Audit & Diagnostic Charts

In [ ]:
def _one_sample_ttest_str(series: pd.Series) -> str:
    vals = pd.Series(series).dropna().values
    if len(vals) < 3 or np.std(vals, ddof=1) == 0:
        return "N/A (n < 3)"
    t_stat, p_val = stats.ttest_1samp(vals, 0.0)
    sig = "*" if p_val < 0.05 else ("†" if p_val < 0.10 else "ns")
    return f"t={t_stat:+.2f}, p={p_val:.4f} ({sig})"


def evaluate_strategy(df: pd.DataFrame, signal_col: str) -> dict:
    buys = df[df[signal_col].str.contains("BUY", na=False)]
    shorts = df[df[signal_col].str.contains("SHORT", na=False)]
    active = pd.concat([buys, shorts]).copy()
    if active.empty:
        return {"Active Trades": "0"}

    direction = active[signal_col].str.contains("BUY", na=False).map({True: 1, False: -1})
    active["dir_T1"] = active["actual_T1_ret"] * direction
    active["alpha_T1"] = active["excess_T1"] * direction
    active["dir_T5"] = active["actual_T5_ret"] * direction
    active["alpha_T5"] = active["excess_T5"] * direction

    w1 = int((active["dir_T1"] > 0).sum())
    w5 = int((active["dir_T5"] > 0).sum())
    ir_t5 = active["alpha_T5"].mean() / max(1e-6, active["alpha_T5"].std(ddof=1)) if len(active) > 1 else 0.0

    return {
        "Active Trades": f"{len(active)} ({len(buys)} Buy, {len(shorts)} Short)",
        "T+1 Win Rate": f"{(w1 / len(active)) * 100:.1f}% ({w1}/{len(active)})",
        "Avg. T+1 Alpha (vs SPY)": f"{active['alpha_T1'].mean():+.2f}%",
        "T+5 Win Rate": f"{(w5 / len(active)) * 100:.1f}% ({w5}/{len(active)})",
        "Avg. T+5 Alpha (vs SPY)": f"{active['alpha_T5'].mean():+.2f}%",
        "T+5 Info Ratio (Mean/Std)": f"{ir_t5:+.2f}",
        "T+5 Alpha t-test (H0: μ=0)": _one_sample_ttest_str(active["alpha_T5"]),
    }


scorecard = pd.DataFrame({
    "Tier 1: LM Lexicon": evaluate_strategy(test_df, "sig_lm_lexicon"),
    "Tier 2a: FinBERT Whole": evaluate_strategy(test_df, "sig_finbert_whole"),
    "Tier 2b: FinBERT Q&A": evaluate_strategy(test_df, "sig_finbert_qa"),
    "Tier 3a: Full LogReg (Tail)": evaluate_strategy(test_df, "sig_logreg_full"),
    "Tier 3b: Compact LogReg (Tail)": evaluate_strategy(test_df, "sig_logreg_compact"),
    "Tier 3c: Full Ridge (Tail)": evaluate_strategy(test_df, "sig_ridge_full"),
})
scorecard.to_csv(OUT_DIR / "oos_strategy_scorecard.csv")
print(f"=== BLIND OUT-OF-SAMPLE SCORECARD (Test n={len(test_df)} | Tail={int(TAIL_QUANTILE*100)}%) ===")
display(scorecard)

# --- DOWNSIDE RISK AUDIT (Full Sample) ---
risk_score = chrono_df["evasion_rate_pct"].rank(pct=True) + chrono_df["lm_uncertainty_rate"].rank(pct=True)
high_risk = chrono_df[risk_score >= risk_score.quantile(0.75)]
low_risk = chrono_df[risk_score <= risk_score.quantile(0.25)]

downside_df = pd.DataFrame([
    {
        "Cohort (Full Sample)": "High Evasion + LM Uncertainty (Top 25%)",
        "Calls (n)": len(high_risk),
        "Big-Drop Rate (<-3% vs SPY)": f"{(high_risk['excess_T5'] < -3.0).mean() * 100:.1f}%",
        "Avg Excess T+5 Return": f"{high_risk['excess_T5'].mean():+.2f}%",
        "Avg |T+5 Volatility|": f"{high_risk['excess_T5'].abs().mean():.2f}%",
    },
    {
        "Cohort (Full Sample)": "Low Evasion + LM Uncertainty (Bottom 25%)",
        "Calls (n)": len(low_risk),
        "Big-Drop Rate (<-3% vs SPY)": f"{(low_risk['excess_T5'] < -3.0).mean() * 100:.1f}%",
        "Avg Excess T+5 Return": f"{low_risk['excess_T5'].mean():+.2f}%",
        "Avg |T+5 Volatility|": f"{low_risk['excess_T5'].abs().mean():.2f}%",
    },
])
downside_df.to_csv(OUT_DIR / "downside_risk_audit.csv", index=False)
print("\n=== DOWNSIDE RISK AUDIT (High vs. Low Evasion + LM Uncertainty) ===")
display(downside_df)

print("\n=== FEATURE CORRELATIONS WITH EXCESS RETURN (Full Sample) ===")
display(chrono_df[FULL_FEATURES + ["excess_T1", "excess_T5"]].corr()[["excess_T1", "excess_T5"]].round(4))

=== BLIND OUT-OF-SAMPLE SCORECARD (Test n=1200 | Tail=20%) ===


,Tier 1: LM Lexicon,Tier 2a: FinBERT Whole,Tier 2b: FinBERT Q&A,Tier 3a: Full LogReg (Tail),Tier 3b: Compact LogReg (Tail),Tier 3c: Full Ridge (Tail)
Active Trades,"742 (456 Buy, 286 Short)","775 (537 Buy, 238 Short)","762 (356 Buy, 406 Short)","565 (276 Buy, 289 Short)","589 (228 Buy, 361 Short)","545 (188 Buy, 357 Short)"
T+1 Win Rate,50.3% (373/742),49.4% (383/775),47.6% (363/762),52.0% (294/565),52.5% (309/589),52.8% (288/545)
Avg. T+1 Alpha (vs SPY),-0.01%,-0.06%,-0.14%,+0.02%,+0.00%,-0.08%
T+5 Win Rate,51.9% (385/742),51.5% (399/775),48.7% (371/762),49.4% (279/565),47.4% (279/589),47.9% (261/545)
Avg. T+5 Alpha (vs SPY),+0.09%,-0.05%,-0.24%,+0.16%,-0.10%,+0.11%
T+5 Info Ratio (Mean/Std),+0.02,-0.01,-0.06,+0.04,-0.02,+0.03
T+5 Alpha t-test (H0: μ=0),"t=+0.54, p=0.5905 (ns)","t=-0.36, p=0.7202 (ns)","t=-1.52, p=0.1280 (ns)","t=+0.85, p=0.3967 (ns)","t=-0.54, p=0.5906 (ns)","t=+0.59, p=0.5566 (ns)"



=== DOWNSIDE RISK AUDIT (High vs. Low Evasion + LM Uncertainty) ===


,Cohort (Full Sample),Calls (n),Big-Drop Rate (<-3% vs SPY),Avg Excess T+5 Return,Avg |T+5 Volatility|
0,High Evasion + LM Uncertainty (Top 25%),1000,15.7%,-0.09%,2.56%
1,Low Evasion + LM Uncertainty (Bottom 25%),1000,17.5%,-0.16%,2.61%



=== FEATURE CORRELATIONS WITH EXCESS RETURN (Full Sample) ===


,excess_T1,excess_T5
finbert_score,-0.0264,-0.0265
finbert_qa_score,-0.0322,-0.0487
finbert_tone_delta,0.0006,-0.0114
lm_net_score,-0.0217,-0.0181
lm_uncertainty_rate,0.0288,0.0356
qa_neg_pct,0.0153,0.0453
evasion_rate_pct,-0.0121,-0.0151
pre_5d_excess_ret,-0.0023,0.0152
excess_T1,1.0000,0.5997
excess_T5,0.5997,1.0000


In [ ]:
# --- 6.2 INTERACTIVE DIAGNOSTIC CHARTS (LEARNED WEIGHTS + OOS CUMULATIVE ALPHA) ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Learned Standardized ML Weights (Full Logistic Regression)",
        f"Out-of-Sample Cumulative T+5 Excess Alpha (Test n={len(test_df)})",
    ),
    horizontal_spacing=0.14,
)

# Panel 1: Learned Feature Weights
w_items = sorted(w_logreg_full.items(), key=lambda x: x[1])
feat_names = [k for k, _ in w_items]
feat_vals = [v for _, v in w_items]
fig.add_trace(
    go.Bar(
        x=feat_vals,
        y=feat_names,
        orientation="h",
        marker_color=["#16A34A" if v >= 0 else "#DC2626" for v in feat_vals],
        text=[f"{v:+.3f}" for v in feat_vals],
        textposition="outside",
        name="LogReg Weight",
        showlegend=False,
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="gray", row=1, col=1)

# Panel 2: Cumulative OOS Excess Alpha Curves (sum of trade excess alphas over time)
strategies_to_plot = [
    ("sig_lm_lexicon", "Tier 1: LM Lexicon", "#94A3B8"),
    ("sig_finbert_whole", "Tier 2: FinBERT Whole", "#2563EB"),
    ("sig_logreg_full", "Tier 3a: Full LogReg (Tail)", "#16A34A"),
    ("sig_logreg_compact", "Tier 3b: Compact LogReg (Tail)", "#D97706"),
]
for col, label, color in strategies_to_plot:
    dir_series = test_df[col].apply(lambda s: 1 if "BUY" in str(s) else (-1 if "SHORT" in str(s) else 0))
    trade_alpha = test_df["excess_T5"] * dir_series
    cum_alpha = trade_alpha.cumsum()
    fig.add_trace(
        go.Scatter(
            x=test_df["earnings_dt"],
            y=cum_alpha,
            mode="lines+markers",
            name=label,
            line=dict(color=color, width=2.5),
            marker=dict(size=5),
        ),
        row=1, col=2,
    )
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_layout(height=480, template="plotly_white", margin=dict(t=60, b=40))
fig.update_xaxes(title_text="Standardized Coefficient (Train)", row=1, col=1)
fig.update_xaxes(title_text="Out-of-Sample Earnings Date", row=1, col=2)
fig.update_yaxes(title_text="Cumulative T+5 Excess Alpha (%)", row=1, col=2)

fig.write_html(str(OUT_DIR / "backtest_ml_diagnostics.html"))
fig.show()